# 06_00 Making it talk (overview)

*📖 Book: sections 4.7, 5.1.1 and 5.3 (text generation); the MCP part is our own*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → [model writes] → MCP tool
> ```
>
> You are here: **model writes**
>
> Project files this connects to:
>
> - `src/homer_ai/homer_gpt.pt`
> - `src/homer_ai/homer_mcp_server.py`
>
> **This notebook:** The model can only guess ONE next letter, so how does it write a whole line? And how can other AI apps, like Claude, ask our model for Greek?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How a model writes: guess one letter, add it, guess again
- Why the plainest way of writing gets stuck, and what fixes it
- How other AI apps, like Claude, can use our model through MCP
- Which notebook to open for each idea

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **generation** | making new text: the model guesses one letter, we add it to the end, and it guesses again | `ἄνδρα` → `ἄνδρα τοι …` |
| **autoregressive** | feeding the model's own output back in as its next input (auto = self) | the letter it just wrote is read in the next round |
| **prompt** | the text you give a model to continue | `ἄνδρα` |
| **tool** | a function an AI app is allowed to call, with a name, a description and arguments | `consult_homer_llm` |
| **MCP** | Model Context Protocol: the standard way for an AI app to find and call tools | like one plug shape that fits every socket |
| **MCP server** | a small program that offers tools through MCP | `homer_mcp_server.py` |
| **MCP client** | the AI app that connects to MCP servers and calls their tools | Claude Code |

## 3. The idea

### ❓ The model only guesses ONE next letter. How does it write a whole line?

You know the row of suggestions above your phone's keyboard? Tap the middle one again and again and the phone writes a sentence by itself, one word at a time, each one picked from what's there so far. Our model writes the same way, and it has the same weakness: keep taking the top suggestion and it soon goes round in circles. The second half of the chapter lends that keyboard out, so Claude can ask our model for a line of Greek the way you'd ask a colleague who's good at exactly one thing. The picture flatters us in one way: your phone suggests whole words and has learned from millions of people, and our model guesses a single letter and has read one book of one poem.

So, how does one guess become a line? By doing it over and over. The model guesses a letter, we add it to the text, and the longer text goes back in for the next guess. Feeding the output back in as the next input is called **autoregressive** (*auto* = self). Here are the first four rounds on the real model, starting from the prompt `ἄνδρα` ("man", the Odyssey's first word). `␣` stands for a space:

```
round 1   reads: ἄνδρα      guess: ␣
round 2   reads: ἄνδρα␣     guess: τ
round 3   reads: ἄνδρα␣τ    guess: ο
round 4   reads: ἄνδρα␣το   guess: ι
written:  ἄνδρα τοι         (τοι: "to you", a small word)
```

So each round the text grows by one letter, and the model reads the longer text.

👉 **Predict first:** the code below runs exactly this on the trained model. Will its picture match?

In [ ]:
import torch

from homer_ai.lessons import load_homer_model

# read-only: we only load the saved model, we never save
tokenizer, model = load_homer_model()  # yours, or else the ready-made one

In [ ]:
if model is not None:
    text = "ἄνδρα"
    for round_number in range(1, 5):
        ids = torch.tensor(tokenizer.encode(text))
        with torch.no_grad():  # we only guess here, we do not learn
            scores = model(ids)[-1]  # [-1]: the guess after the last letter
        best = scores.argmax().item()  # argmax: the position of the biggest score
        guess = tokenizer.decode([best])
        shown_text = text.replace(" ", "␣")  # make the spaces visible
        shown_guess = guess.replace(" ", "␣")
        print(f"round {round_number}   reads: {shown_text:<8}  guess: {shown_guess}")
        text = text + guess
    print("written: ", text)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Same picture, and this time from real guesses. That's all writing is: this one small round, repeated. In [06_01](06_01_writing_greek.ipynb) we build it as a proper function, `write()`, on the real trained model.

</details>

### ❓ Is the best guess always the best choice?

Not for writing. Always taking the single most likely letter is called **greedy** ([01_01](../01_foundations/01_01_a_language_model_without_a_brain.ipynb)), and it falls into loops. The same few letters come out again and again, like `και και και` ("and and and"). Why that happens is in 06_01. The fix, in [06_02](06_02_less_repetition.ipynb), is to roll a weighted dice instead, so likely letters win more often, but not always. The dice has two dials, called temperature and top-k.

### ❓ How can another AI app, like Claude, use our model?

Through **MCP**, the Model Context Protocol. A protocol is an agreed way of talking. Think of one plug shape: every AI app that has the plug can use every tool that has the socket. Our model sits behind an MCP server (`src/homer_ai/homer_mcp_server.py`), a small program that offers our model as a tool, `consult_homer_llm`. An MCP client, such as Claude Code, connects to the server and can call the tool.

```
you ─▶ Claude Code (the client) ─▶ our MCP server ─▶ our model
                                    consult_homer_llm   writes Greek
you ◀─ Claude's answer ◀─ the tool's text ◀──────────┘
```

In [06_03](06_03_the_mcp_server.ipynb) we take the server apart, piece by piece.

*A lot of my AI work runs inside automation tools like n8n, where the model is just one step in a workflow. That's exactly the role our Homer model plays for Claude.*

### ❓ Who runs the tool, Claude or our model?

Neither of them, really. Claude only writes a small text form in **JSON** (plain text for data, with names and values in curly brackets). The program around Claude, the harness, is what runs our tool. The [course map](../00_start_here/00_00_course_map.ipynb) explains the whole chain once, in full, and [06_03](06_03_the_mcp_server.ipynb) shows our server's part of it.

### ❓ Which notebook do I open for each idea?

| Notebook | Question it answers |
|---|---|
| [`06_01_writing_greek.ipynb`](06_01_writing_greek.ipynb) | How do I build `write()`, and why does it loop? |
| [`06_02_less_repetition.ipynb`](06_02_less_repetition.ipynb) | How do temperature and top-k make the writing less repetitive? |
| [`06_03_the_mcp_server.ipynb`](06_03_the_mcp_server.ipynb) | How does our server work, and how do I connect it to Claude Code? |

## 4. Recap

- Writing is: guess one letter, add it, guess again (autoregressive). Greedy picking loops.
- A weighted dice (temperature, top-k) makes the writing less repetitive.
- An MCP server offers our model as a tool. The client's harness runs it, and Claude only writes text.

**Next:** [06_01: Writing Greek](06_01_writing_greek.ipynb).